# Fine-tune `tiny_bert` with LoRA, then encode a query with the result

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/recipes/fine_tune.ipynb)

Built from [`cookbook/recipes/fine_tune/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/fine_tune/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Then fine-tune from a citation graph instead of labelled pairs.

Run with `python cookbook/recipes/fine_tune/example.py`. Exits 0 on
success; seconds on CPU.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import jammi
from jammi_cookbook import fixtures

PAIRS_PATH = fixtures.path("tiny_pairs.csv")
BASE_MODEL = fixtures.model("tiny_bert")
CITATIONS = fixtures.path("tiny_citation_graph")


def main() -> int:
    with tempfile.TemporaryDirectory() as tmp, jammi.connect(f"file://{tmp}") as db:

        # 1. Register the contrastive training pairs.
        db.add_source("training", url=str(PAIRS_PATH), format="csv")

        # 2. Submit the fine-tune job. Defaults are tuned for production
        #    workloads; for the cookbook we keep rank small and run a
        #    single epoch so the example finishes quickly.
        job = db.fine_tune(
            source="training",
            base_model=BASE_MODEL,
            columns=["text_a", "text_b", "score"],
            method="lora",
            task="text_embedding",
            lora_rank=4,
            epochs=1,
        )
        assert job.job_id, "fine_tune returned a job without an id"
        print(f"job_id:    {job.job_id}")

        # 3. Block until the job reaches a terminal state.
        job.wait()

        # 4. Newly-registered model_id follows the jammi:fine-tuned:* shape.
        model_id = job.output_model_id
        assert model_id.startswith("jammi:fine-tuned:"), (
            f"unexpected model_id: {model_id}"
        )
        print(f"model_id:  {model_id}")

        # 5. Encode a query through the fine-tuned model to confirm it
        #    loads end-to-end from the catalog.
        query_vec = db.encode_query(model=model_id, query="quantum computing applications")
        assert len(query_vec) == 32, (
            f"tiny_bert is 32-dim; got {len(query_vec)}-dim from fine-tuned"
        )

        # 6. Fine-tune from a graph instead of labelled pairs: nodes that cite
        #    each other are pulled together. Random walks over the citation
        #    edges sample the positives, the graph's non-neighbours the
        #    negatives. The nodes and edges are two ordinary sources.
        db.add_source("papers", url=str(CITATIONS / "nodes.jsonl"), format="jsonl")
        db.add_source("cites", url=str(CITATIONS / "edges.jsonl"), format="jsonl")
        graph_job = db.fine_tune_graph(
            node_source="papers",
            id_column="id",
            text_column="text",
            edge_source="cites",
            base_model=BASE_MODEL,
            lora_rank=4,
            epochs=1,
            sample_seed=0,
        )
        graph_job.wait()
        print(f"graph-tuned model_id: {graph_job.output_model_id}")
        assert graph_job.output_model_id.startswith("jammi:fine-tuned:")

    print("fine_tune: OK")
    return 0

In [ ]:
assert main() == 0